In [6]:
# Some Imports & Setup from onboarding1 in case I need it
import environments
import lsp
import numpy as np
import os
import math
import gridmap
import sys

# imports from object_search
import matplotlib.pyplot as plt
import procthor
from procthor.simulators import SceneGraphSimulator
import object_search
from object_search.planners import PlanningLoop, OptimisticPlanner, LearnedPlannerFCNN
from pathlib import Path

print("Python Version and System Information")
print(sys.version)

Python Version and System Information
3.10.12 (main, Aug 15 2025, 14:32:43) [GCC 11.4.0]


In [7]:
# configure parameters
def get_args():
    """Configure experiment parameters"""
    args = lambda key: None  # Simple namespace object
    args.save_dir = '/data/my_test1_logs'
    args.current_seed = 0
    args.resolution = 0.05
    args.do_save_video = False
    return args

# Get configuration
args = get_args()
print(f"Experiment configuration:")
print(f"  Save directory: {args.save_dir}")
print(f"  Seed: {args.current_seed}")
print(f"  Grid resolution: {args.resolution}m")
print(f"  Save video: {args.do_save_video}")

Experiment configuration:
  Save directory: /data/my_test1_logs
  Seed: 0
  Grid resolution: 0.05m
  Save video: False


In [ ]:
# object-serach test on optimistic planner
def test_object_search_optimistic_planner():
    '''Test object search in ProcTHOR environment with OptimisticPlanner'''
    
    print("Initializing ProcTHOR environment...")
    thor_interface = procthor.ThorInterface(args)
    known_graph, known_grid, robot_pose, target_obj_info = thor_interface.gen_map_and_poses()

    #known_graph, known_grid, robot_pose, target_obj_info = environments.generate.map_and_poses(args)
    
    #print(f"Target object: {target_obj_info['name']}")
    #print(f"Robot starting pose: ({robot_pose.x:.2f}, {robot_pose.y:.2f})")
    
    # Initialize simulator and components
    simulator = SceneGraphSimulator(known_graph,
                                    args,
                                    target_obj_info,
                                    known_grid,
                                    environments)   # switched out thor_interface
    
    robot = object_search.robot.Robot(robot_pose)
    planner = OptimisticPlanner(target_obj_info, args)
    planning_loop = PlanningLoop(target_obj_info, simulator, robot, args=args, verbose=True)
    
    print("Starting planning loop...")
    # Execute planning loop
    for step_idx, step_data in enumerate(planning_loop):  # Fixed syntax error here
        print(f"Planning step {step_idx}")
        
        # Update planner with current observations
        planner.update(
            step_data['observed_graph'],
            step_data['observed_grid'],
            step_data['subgoals'],
            step_data['robot_pose'])
        
        # Compute and set next subgoal
        chosen_subgoal = planner.compute_selected_subgoal()
        planning_loop.set_chosen_subgoal(chosen_subgoal)
    
    # Compute final metrics
    cost, trajectory = object_search.utils.compute_cost_and_trajectory(
        known_grid, robot.all_poses, args.resolution, use_robot_model=True)
    
    print(f"Planning completed! Total cost: {cost:.1f} meters")
    return known_graph, known_grid, robot_pose, target_obj_info, simulator, robot, cost, trajectory

# Run the experiment
results = test_object_search_optimistic_planner()
known_graph, known_grid, robot_pose, target_obj_info, simulator, robot, cost, trajectory = results

Initializing ProcTHOR environment...


In [ ]:
# let's visualize the test runs

plt.figure(figsize=(16, 12))

known_locations = [known_graph.get_node_name_by_idx(idx) for idx in target_obj_info['container_idxs']]
plt.suptitle(f"Seed: {args.current_seed} | Target object: {target_obj_info['name']}\n"
             f"Known locations: {known_locations}", fontsize=14)

# 1. Scene graph visualization
ax = plt.subplot(221)
plt.title('Whole scene graph')
procthor.plotting.plot_graph(ax, known_graph.nodes, known_graph.edges)

# 2. Graph overlaid on grid
ax = plt.subplot(222)
procthor.plotting.plot_graph_on_grid(ax, known_grid, known_graph)
plt.text(robot_pose.x, robot_pose.y, '+', color='red', size=8, weight='bold')
plt.title('Graph over occupancy grid')

# 3. Top-down view
plt.subplot(223)
top_down_image = simulator.get_top_down_image()
plt.imshow(top_down_image)
plt.title('Top-down view of the scene')
plt.axis('off')

# 4. Robot trajectory
ax = plt.subplot(224)
object_search.plotting.plot_grid_with_robot_trajectory(ax, known_grid, robot.all_poses, trajectory, known_graph)
plt.title(f"Robot Trajectory | Cost: {cost:.1f} meters")

plt.tight_layout()